# Model Comparison with WAIC

WAIC (Widely Applicable Information Criterion) is a Bayesian approach to model comparison that estimates out-of-sample predictive accuracy.

**Key concepts:**
- **WAIC**: Lower is better (like AIC/BIC)
- **p_waic**: Effective number of parameters (penalty for complexity)
- **lppd**: Log pointwise predictive density (model fit)
- **SE**: Standard error for comparison uncertainty

**Formula:** WAIC = -2 × (lppd - p_waic)

In this notebook, we compare polynomial regression models of different complexity:
1. **Linear** (degree 1) - underfitting
2. **Quadratic** (degree 2) - the true model
3. **Degree 5** - overfitting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.typing import NDArray

import minibayes as mb
from minibayes import dist, viz

---
## 1. Generate Synthetic Data

True model: $y = 1 + 2x + 0.5x^2 + \epsilon$ where $\epsilon \sim N(0, 0.5)$

In [ ]:
rng = np.random.default_rng(42)
n = 50
x = np.linspace(-2, 2, n)
y_true = 1 + 2 * x + 0.5 * x**2
y = y_true + rng.normal(0, 0.5, n)

print(f"Data: {n} observations")
print(f"True model: y = 1 + 2x + 0.5x²")

In [ ]:
with viz.style():
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(x, y, color=viz.PALETTE["terracotta"], alpha=0.7, s=40, label="Observed data")
    ax.plot(x, y_true, color=viz.PALETTE["sage"], lw=2, ls="--", label="True function")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("Synthetic Polynomial Data")
    ax.legend()
    plt.show()

---
## 2. Define Competing Models

We define three polynomial regression models with increasing complexity.

In [ ]:
# Model 1: Linear (degree 1) - UNDERFITTING
def priors_linear(p):
    p("a", dist.Normal(0, 5))
    p("b", dist.Normal(0, 5))
    p("sigma", dist.HalfNormal(2))


def log_lik_linear(params, data) -> NDArray[np.float64]:
    x, y = data
    mu = params["a"] + params["b"] * x
    return dist.Normal(mu, params["sigma"]).log_prob(y)


model_linear = mb.Model(priors=priors_linear, log_likelihood=log_lik_linear)
print(f"Linear model: {model_linear.param_names}")

In [ ]:
# Model 2: Quadratic (degree 2) - TRUE MODEL
def priors_quad(p):
    p("a", dist.Normal(0, 5))
    p("b", dist.Normal(0, 5))
    p("c", dist.Normal(0, 5))
    p("sigma", dist.HalfNormal(2))


def log_lik_quad(params, data) -> NDArray[np.float64]:
    x, y = data
    mu = params["a"] + params["b"] * x + params["c"] * x**2
    return dist.Normal(mu, params["sigma"]).log_prob(y)


model_quad = mb.Model(priors=priors_quad, log_likelihood=log_lik_quad)
print(f"Quadratic model: {model_quad.param_names}")

In [ ]:
# Model 3: Degree 5 polynomial - OVERFITTING
def priors_poly5(p):
    p("a", dist.Normal(0, 5))
    p("b", dist.Normal(0, 5))
    p("c", dist.Normal(0, 5))
    p("d", dist.Normal(0, 5))
    p("e", dist.Normal(0, 5))
    p("f", dist.Normal(0, 5))
    p("sigma", dist.HalfNormal(2))


def log_lik_poly5(params, data) -> NDArray[np.float64]:
    x, y = data
    mu = (
        params["a"]
        + params["b"] * x
        + params["c"] * x**2
        + params["d"] * x**3
        + params["e"] * x**4
        + params["f"] * x**5
    )
    return dist.Normal(mu, params["sigma"]).log_prob(y)


model_poly5 = mb.Model(priors=priors_poly5, log_likelihood=log_lik_poly5)
print(f"Degree-5 model: {model_poly5.param_names}")

---
## 3. Fit All Models

In [ ]:
data = (x, y)

print("Fitting Linear model...")
result_linear = mb.sample(
    model_linear, data=data, num_samples=2000, num_warmup=500, sampler="ensemble", seed=42
)

print("Fitting Quadratic model...")
result_quad = mb.sample(model_quad, data=data, num_samples=2000, num_warmup=500, sampler="ensemble", seed=42)

print("Fitting Degree-5 model...")
result_poly5 = mb.sample(model_poly5, data=data, num_samples=2000, num_warmup=500, sampler="ensemble", seed=42)

print("\nAll models fitted!")

---
## 4. Compute WAIC

Use the `result.waic(model, data)` convenience method to compute WAIC for each model.

In [ ]:
waic_linear = result_linear.waic(model_linear, data)
waic_quad = result_quad.waic(model_quad, data)
waic_poly5 = result_poly5.waic(model_poly5, data)

print("Model Comparison Results")
print("=" * 60)
print(f"{'Model':<15} {'WAIC':>10} {'p_waic':>10} {'lppd':>10} {'SE':>10}")
print("-" * 60)
print(f"{'Linear':<15} {waic_linear.waic:>10.1f} {waic_linear.p_waic:>10.1f} {waic_linear.lppd:>10.1f} {waic_linear.se:>10.1f}")
print(f"{'Quadratic':<15} {waic_quad.waic:>10.1f} {waic_quad.p_waic:>10.1f} {waic_quad.lppd:>10.1f} {waic_quad.se:>10.1f}")
print(f"{'Poly-5':<15} {waic_poly5.waic:>10.1f} {waic_poly5.p_waic:>10.1f} {waic_poly5.lppd:>10.1f} {waic_poly5.se:>10.1f}")
print("=" * 60)

# Find best model
waics = {"Linear": waic_linear, "Quadratic": waic_quad, "Poly-5": waic_poly5}
best = min(waics.items(), key=lambda x: x[1].waic)
print(f"\nBest model: {best[0]} (lowest WAIC)")

---
## 5. Visualize Comparison with `plot_compare()`

The `viz.plot_compare()` function creates a forest-style plot showing:
- WAIC point estimates with ±2 SE error bars
- Models ranked by WAIC (best at top)
- Delta-WAIC from best model

In [ ]:
waic_results = {
    "Linear": waic_linear,
    "Quadratic": waic_quad,
    "Poly-5": waic_poly5,
}

fig = viz.plot_compare(waic_results)
plt.show()

---
## 6. Interpretation Guidelines

### Reading the Results

- **Best model**: Lowest WAIC wins (Quadratic in this case)
- **p_waic**: Effective number of parameters
  - Should be close to actual parameter count for well-specified models
  - Can exceed parameter count for overfitting models
- **Standard errors**: Use for significance testing
  - If ΔWAIC < 2×SE, models are statistically similar
  - Larger differences are more meaningful

### Why Quadratic Wins

1. **Linear underfits**: High WAIC because it can't capture the curvature
2. **Quadratic fits well**: Matches the true data-generating process
3. **Poly-5 overfits**: May have similar lppd but higher p_waic penalty

---
## 7. Posterior Predictive Comparison

Let's visualize the fitted curves from all three models.

In [ ]:
x_plot = np.linspace(-2.5, 2.5, 100)


def pred_linear(params, rng):
    return {"y": params["a"] + params["b"] * x_plot}


def pred_quad(params, rng):
    return {"y": params["a"] + params["b"] * x_plot + params["c"] * x_plot**2}


def pred_poly5(params, rng):
    return {
        "y": params["a"]
        + params["b"] * x_plot
        + params["c"] * x_plot**2
        + params["d"] * x_plot**3
        + params["e"] * x_plot**4
        + params["f"] * x_plot**5
    }


ppc_linear = result_linear.predict(pred_linear, num_samples=200, seed=42)
ppc_quad = result_quad.predict(pred_quad, num_samples=200, seed=42)
ppc_poly5 = result_poly5.predict(pred_poly5, num_samples=200, seed=42)

In [ ]:
with viz.style():
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

    models_data = [
        ("Linear", ppc_linear["y"], waic_linear),
        ("Quadratic", ppc_quad["y"], waic_quad),
        ("Poly-5", ppc_poly5["y"], waic_poly5),
    ]

    for ax, (name, ppc, waic_res) in zip(axes, models_data):
        # Plot posterior samples
        for i in range(0, 200, 5):
            ax.plot(x_plot, ppc[i], color=viz.PALETTE["blue"], alpha=0.05, lw=1)

        # Plot mean prediction
        ax.plot(x_plot, ppc.mean(axis=0), color=viz.PALETTE["blue"], lw=2, label="Posterior mean")

        # Plot true function
        y_true_plot = 1 + 2 * x_plot + 0.5 * x_plot**2
        ax.plot(x_plot, y_true_plot, color=viz.PALETTE["sage"], lw=2, ls="--", label="True")

        # Plot data
        ax.scatter(x, y, color=viz.PALETTE["terracotta"], alpha=0.6, s=30, zorder=5)

        ax.set_xlabel("x")
        ax.set_title(f"{name}\nWAIC={waic_res.waic:.1f}, p_waic={waic_res.p_waic:.1f}")
        if ax == axes[0]:
            ax.set_ylabel("y")
            ax.legend(loc="upper left", fontsize=8)

    plt.suptitle("Posterior Predictions by Model Complexity", y=1.02)
    plt.tight_layout()
    plt.show()

---
## 8. Pointwise WAIC Analysis

The `pointwise` attribute of WAICResult contains per-observation WAIC contributions. This helps identify:
- **Influential observations**: High pointwise WAIC = hard to predict
- **Outliers**: Observations where the model struggles

In [ ]:
with viz.style():
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

    models_data = [
        ("Linear", waic_linear),
        ("Quadratic", waic_quad),
        ("Poly-5", waic_poly5),
    ]

    for ax, (name, waic_res) in zip(axes, models_data):
        ax.bar(range(n), waic_res.pointwise, color=viz.PALETTE["blue"], alpha=0.7)
        ax.axhline(waic_res.pointwise.mean(), color=viz.PALETTE["terracotta"], ls="--", lw=2)
        ax.set_xlabel("Observation index")
        ax.set_title(f"{name}")
        if ax == axes[0]:
            ax.set_ylabel("Pointwise WAIC")

    plt.suptitle("Pointwise WAIC Contributions (dashed = mean)", y=1.02)
    plt.tight_layout()
    plt.show()

# Identify most influential observations for the best model
top_3 = np.argsort(waic_quad.pointwise)[-3:]
print(f"\nMost influential observations for Quadratic model: {top_3}")
print(f"These observations contribute most to the model's WAIC.")

---
## Summary

### WAIC API

```python
# Convenience method on result
waic_result = result.waic(model, data)

# Or standalone function
waic_result = mb.waic(result, model, data)
```

### WAICResult Fields

| Field | Description |
|-------|-------------|
| `waic` | Total WAIC (lower = better predictive accuracy) |
| `p_waic` | Effective number of parameters |
| `lppd` | Log pointwise predictive density (fit) |
| `se` | Standard error of WAIC estimate |
| `pointwise` | Per-observation contributions (shape: n_obs) |

### Visualization

```python
# Compare multiple models
waic_results = {
    "Model A": result_a.waic(model_a, data),
    "Model B": result_b.waic(model_b, data),
}
viz.plot_compare(waic_results)
```

### When to Use WAIC

- **Model selection**: Choose between competing models
- **Complexity assessment**: Check if your model is overfitting
- **Influential observations**: Identify outliers or unusual data points

### References

- Gelman et al. (2014) *Bayesian Data Analysis*, 3rd ed., Chapter 7
- Vehtari et al. (2017) *Practical Bayesian model evaluation using leave-one-out cross-validation and WAIC*